# Cheetah model calibration

Learn the transverse offsets of three quadrupoles from screen images and centroids at nine scan settings, using independent clones of an in-process LUMEcheetah model. Compare a Lightning gradient-descent fit with Bayesian inference using Pyro SVI.

In [ ]:
import os
from copy import deepcopy

os.environ["CUDA_VISIBLE_DEVICES"] = ""

import matplotlib.pyplot as plt
import pandas as pd
import torch
from lightning.pytorch import Trainer

from lume_applications.model_calibration import (
    CalibrationModule,
    get_trainable_parameters,
)

torch.set_num_threads(1)

## Prepare example data

Build the Q1-D1-Q2-D2-Q3-D3-SCR1 beamline with `create_example_model`. Generate screen images and beam centroids with horizontal/vertical offsets of `(2, -3)` mm for Q1, `(-1, 3)` mm for Q2, and `(0, 0)` mm for Q3.

The particle seed is fixed by the model factory. Q1 and Q2 follow the specified strength scans, while Q3 uses nine random strengths. `all_data` contains one row per setting, with three input-strength columns and the screen image and two centroid outputs. Screen coordinates and centroid values are in meters before plotting.

In [ ]:
from example_model import create_example_model

reference_model = create_example_model()

true_offsets = torch.tensor([[2e-3, -3e-3], [-1e-3, 3e-3], [0.0, 0.0]])
reference_model.simulator.segment.Q1.misalignment = true_offsets[0].clone()
reference_model.simulator.segment.Q2.misalignment = true_offsets[1].clone()
reference_model.simulator.segment.Q3.misalignment = true_offsets[2].clone()

# create dataset by scanning quadrupole strengths and recording screen images
q1_strengths = torch.linspace(-2.0, 2.0, 9)
q2_strengths = torch.linspace(-1.0, 2.0, 9).flipud()
q3_strengths = torch.randn(9)

inputs = [
    {"Q1_k": q1_strengths[j], "Q2_k": q2_strengths[j], "Q3_k": q3_strengths[j]}
    for j in range(len(q1_strengths))
]
input_data = pd.DataFrame(inputs)

data = []
for j in range(len(inputs)):
    reference_model.set(inputs[j])
    data.append(
        reference_model.get(["screen_image", "screen_centroid_x", "screen_centroid_y"])
    )

output_data = pd.DataFrame(data)
all_data = pd.concat([input_data, output_data], axis=1)
all_data

In [ ]:
screen = reference_model.simulator.segment.SCR1
screen_x_mm, screen_y_mm = [
    centers.detach().cpu().numpy() * 1e3 for centers in screen.pixel_bin_centers
]
scan_images = [
    torch.as_tensor(image).detach().cpu().numpy() for image in all_data["screen_image"]
]
scan_columns = min(3, len(scan_images))
scan_rows = (len(scan_images) + scan_columns - 1) // scan_columns
scan_max = max(float(image.max()) for image in scan_images) or 1.0
scan_figure, scan_axes = plt.subplots(
    scan_rows,
    scan_columns,
    figsize=(4 * scan_columns, 4 * scan_rows),
    layout="constrained",
    squeeze=False,
    sharex=True,
    sharey=True,
)
for axis, sample, image in zip(
    scan_axes.flat,
    all_data.itertuples(index=False),
    scan_images,
):
    scan_artist = axis.pcolormesh(
        screen_x_mm,
        screen_y_mm,
        image,
        shading="nearest",
        cmap="viridis",
        vmin=0,
        vmax=scan_max,
    )
    screen_bounds = scan_artist.get_datalim(axis.transData)
    axis.set_xlim(screen_bounds.x0, screen_bounds.x1)
    axis.set_ylim(screen_bounds.y0, screen_bounds.y1)
    centroid_x_mm = float(sample.screen_centroid_x) * 1e3
    centroid_y_mm = float(sample.screen_centroid_y) * 1e3
    axis.plot(
        centroid_x_mm,
        centroid_y_mm,
        marker="+",
        linestyle="none",
        color="tab:red",
        markersize=12,
        markeredgewidth=2,
        label=f"Centroid ({centroid_x_mm:+.2f}, {centroid_y_mm:+.2f}) mm",
    )
    axis.set_aspect("equal")
    axis.legend(loc="upper right", fontsize=7)
    axis.set_title(
        f"Q1_k = {float(sample.Q1_k):+.2f}\n"
        f"Q2_k = {float(sample.Q2_k):+.2f}\n"
        f"Q3_k = {float(sample.Q3_k):+.2f}"
    )
for axis in scan_axes.flat[len(scan_images) :]:
    axis.set_visible(False)

scan_figure.suptitle(r"Quadrupole scan (strengths in $\mathrm{m}^{-2}$)")
scan_figure.supxlabel("Horizontal position (mm)")
scan_figure.supylabel("Vertical position (mm)")
scan_figure.colorbar(
    scan_artist,
    ax=list(scan_axes.flat[: len(scan_images)]),
    label="Screen signal",
)
plt.show()

## Run the Lightning calibration workflow

Stack the `Q1_k`, `Q2_k`, and `Q3_k` columns of `all_data` as training inputs, and `screen_image`, `screen_centroid_x`, and `screen_centroid_y` as targets. Each stacked tensor has one leading entry per DataFrame row. Training data are detached copies; `all_data` is not modified.

Clone the reference model and register a two-component offset parameter for each of the three quadrupoles, starting from zero. Do not call `reset()` while fitting: it replaces the segment and discards the parameter objects held by the optimizer.

`CalibrationModule` supplies one full-scan batch per epoch. Lightning's `Trainer` handles gradient clearing, backpropagation, and optimizer updates. `max_steps=300` gives 300 full-scan Adam updates with default gradient accumulation. Logging and checkpoint files are disabled.

The loss averages normalized MSE equally over scan points and the three outputs. Two-dimensional screen images are divided by a fixed target-image maximum; scalar centroids are divided by `centroid_scale = 1e-3` meters (1 mm). This makes residuals dimensionless; changing these scales changes the relative weight of images and centroids. `calibration.loss_history` records the loss before each update and one fresh final evaluation. Ground-truth offsets are used only for comparison. Re-running the following cell starts a fresh fit.

In [ ]:
if all_data.empty:
    raise ValueError("all_data must contain at least one training sample")

training_inputs = {
    name: torch.stack(
        [torch.as_tensor(value).detach().clone() for value in all_data[name]]
    )
    for name in ("Q1_k", "Q2_k", "Q3_k")
}
training_targets = {
    name: torch.stack(
        [torch.as_tensor(value).detach().clone() for value in all_data[name]]
    )
    for name in ("screen_image", "screen_centroid_x", "screen_centroid_y")
}

# Create a fitting model by copying the reference model
fitting_model = deepcopy(reference_model)

# Define the offset parameter for the fitting model
q1_offset = torch.nn.Parameter(
    torch.zeros_like(fitting_model.simulator.segment.Q1.misalignment)
)
fitting_model.simulator.segment.Q1.misalignment = q1_offset
q2_offset = torch.nn.Parameter(
    torch.zeros_like(fitting_model.simulator.segment.Q2.misalignment)
)
fitting_model.simulator.segment.Q2.misalignment = q2_offset
q3_offset = torch.nn.Parameter(
    torch.zeros_like(fitting_model.simulator.segment.Q3.misalignment)
)
fitting_model.simulator.segment.Q3.misalignment = q3_offset

optimizer = torch.optim.Adam(get_trainable_parameters(fitting_model), lr=1e-4)
image_scale = training_targets["screen_image"].abs().max()
centroid_scale = 1e-3
if not torch.isfinite(image_scale) or image_scale <= 0:
    raise ValueError("Target image scale must be finite and positive")


def calibration_loss(prediction, target):
    scale = image_scale if target.ndim == 2 else centroid_scale
    return torch.nn.functional.mse_loss(prediction / scale, target / scale)


loss_weights = {
    "screen_image": 1.0,
    "screen_centroid_x": 1.0,
    "screen_centroid_y": 1.0,
}
calibration = CalibrationModule(
    fitting_model,
    training_inputs,
    training_targets,
    optimizer,
    loss_fn=calibration_loss,
    loss_weights=loss_weights,
)
trainer = Trainer(
    accelerator="cpu",
    devices=1,
    max_steps=500,
    max_epochs=-1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
)
trainer.fit(calibration)
history = calibration.loss_history

print(f"True offset [mm]:   {(true_offsets * 1e3).tolist()}")
print(
    f"Fitted offset [mm]: {(q1_offset.detach() * 1e3).tolist()}, {(q2_offset.detach() * 1e3).tolist()}, {(q3_offset.detach() * 1e3).tolist()}"
)
print(f"Loss: {history[0]:.6g} -> {history[-1]:.6g}")
print(f"Loss reduction: {100 * (1 - history[-1] / history[0]):.3f}%")

## Inspect and visualize results
Show the loss curve and a comparison between training images and predicted images. 

In [ ]:
loss_figure, loss_axis = plt.subplots(figsize=(7, 4), layout="constrained")
loss_steps = range(len(calibration.loss_history))
loss_axis.semilogy(
    loss_steps,
    calibration.loss_history,
    color="black",
    linewidth=2,
    label="Total",
)
for target_name, target_history in calibration.target_loss_history.items():
    loss_axis.semilogy(loss_steps, target_history, label=target_name)
loss_axis.set(
    xlabel="Optimizer steps",
    ylabel="Summed normalized loss",
    title="Offset calibration",
)
loss_axis.legend()
loss_axis.grid(alpha=0.25)
plt.show()

In [ ]:
sample_count = len(training_inputs["Q1_k"])
predictions = {name: [] for name in training_targets}
with torch.no_grad():
    for sample_index in range(sample_count):
        fitting_model.set(
            {
                name: values[sample_index].clone()
                for name, values in training_inputs.items()
            }
        )
        outputs = fitting_model.get(list(predictions))
        for name in predictions:
            predictions[name].append(outputs[name].detach().clone())
predictions = {name: torch.stack(values) for name, values in predictions.items()}
predicted_images = predictions["screen_image"]

comparison_x_mm, comparison_y_mm = [
    centers.detach().cpu().numpy() * 1e3
    for centers in fitting_model.simulator.segment.SCR1.pixel_bin_centers
]
shared_max = (
    max(
        float(training_targets["screen_image"].max()),
        float(predicted_images.max()),
    )
    or 1.0
)
comparison_figure, comparison_axes = plt.subplots(
    2,
    sample_count,
    figsize=(3 * sample_count, 7),
    layout="constrained",
    squeeze=False,
    sharex=True,
    sharey=True,
)
for sample_index in range(sample_count):
    for row_index, outputs in enumerate((training_targets, predictions)):
        axis = comparison_axes[row_index, sample_index]
        image = outputs["screen_image"][sample_index].detach().cpu().numpy()
        comparison_artist = axis.pcolormesh(
            comparison_x_mm,
            comparison_y_mm,
            image,
            shading="nearest",
            cmap="viridis",
            vmin=0,
            vmax=shared_max,
        )
        screen_bounds = comparison_artist.get_datalim(axis.transData)
        axis.set_xlim(screen_bounds.x0, screen_bounds.x1)
        axis.set_ylim(screen_bounds.y0, screen_bounds.y1)
        centroid_x_mm = float(outputs["screen_centroid_x"][sample_index]) * 1e3
        centroid_y_mm = float(outputs["screen_centroid_y"][sample_index]) * 1e3
        axis.plot(
            centroid_x_mm,
            centroid_y_mm,
            marker="+",
            linestyle="none",
            color="tab:red",
            markersize=12,
            markeredgewidth=2,
            label=f"({centroid_x_mm:+.2f}, {centroid_y_mm:+.2f}) mm",
        )
        axis.set_aspect("equal")
        axis.legend(loc="upper right", fontsize=7)
    comparison_axes[0, sample_index].set_title(
        "\n".join(
            f"{name} = {float(training_inputs[name][sample_index]):+.2f}"
            for name in ("Q1_k", "Q2_k", "Q3_k")
        )
    )
comparison_axes[0, 0].set_ylabel("Ground truth")
comparison_axes[1, 0].set_ylabel("Prediction")
comparison_figure.suptitle(
    r"Ground truth and calibrated simulation: centroid crosses; strengths in $\mathrm{m}^{-2}$"
)
comparison_figure.supxlabel("Horizontal position (mm)")
comparison_figure.supylabel("Vertical position (mm)")
comparison_figure.colorbar(
    comparison_artist,
    ax=list(comparison_axes.flat),
    label="Screen signal",
)
plt.show()

## Bayesian misalignment inference with Pyro SVI

This example needs only the imports, `reference_model`, `true_offsets`, and `all_data` from the reference scan above, not the Lightning fit. It stacks detached copies of `Q1_k`, `Q2_k`, and `Q3_k` as `svi_inputs`, and `screen_image`, `screen_centroid_x`, and `screen_centroid_y` as `svi_targets`. The DataFrame and reference model are not modified.

`BayesianModel` maps three exposed Pyro attributes to the Q1, Q2, and Q3 misalignment tensors of an independent model clone. Each attribute is replaced with a two-component `PyroSample` prior: independent zero-centered Normal offsets with 3 mm standard deviation. All six offsets are inferred jointly, with one draw held fixed across every scan point in each likelihood evaluation. Ground-truth offsets are used only for comparison, not prior initialization.

The likelihood assumes independent Gaussian errors: image-pixel standard deviation is 5% of the maximum absolute target image signal, and each centroid has standard deviation `5e-5` m (0.05 mm). These are illustrative assumptions for noiseless synthetic data, not estimated instrument errors. Image noise uses image-intensity units; centroid noise uses meters. The likelihood sums all pixel and centroid log probabilities, unlike the equally averaged output losses in the Lightning example. It ignores correlations between pixels and between images and centroids; changing the assumed noise changes both the fit's weighting and posterior uncertainty.

`infer_model_svi` fits an `AutoNormal` mean-field posterior with an isolated Pyro parameter store. The fit uses 600 SVI steps and returns 2,000 draws for each two-component latent. Location and log-scale parameters use different learning rates because offsets are in meters. `offset_samples_mm` has shape `(draws, quadrupoles, coordinates)` in Q1/Q2/Q3 and horizontal/vertical order. The six marginal plots show the approximate 95% intervals. Mean-field inference cannot represent posterior correlations between offsets and may underestimate uncertainty.

In [ ]:
import pyro
import pyro.distributions as dist
from pyro.nn import PyroSample

from lume_applications.model_calibration.svi import BayesianModel, infer_model_svi

if all_data.empty:
    raise ValueError("all_data must contain at least one training sample")

svi_quadrupoles = ("Q1", "Q2", "Q3")
svi_inputs = {
    name: torch.stack(
        [torch.as_tensor(value).detach().clone() for value in all_data[name]]
    )
    for name in ("Q1_k", "Q2_k", "Q3_k")
}
svi_targets = {
    name: torch.stack(
        [torch.as_tensor(value).detach().clone() for value in all_data[name]]
    )
    for name in ("screen_image", "screen_centroid_x", "screen_centroid_y")
}
svi_parameter_paths = {
    f"{name.lower()}_misalignment": f"simulator.segment.{name}.misalignment"
    for name in svi_quadrupoles
}
bayesian_model = BayesianModel(deepcopy(reference_model), svi_parameter_paths)
for quadrupole_name, latent_name in zip(svi_quadrupoles, svi_parameter_paths):
    offset_template = getattr(
        reference_model.simulator.segment, quadrupole_name
    ).misalignment
    setattr(
        bayesian_model,
        latent_name,
        PyroSample(
            dist.Normal(
                torch.zeros_like(offset_template),
                torch.full_like(offset_template, 3e-3),
            ).to_event(1)
        ),
    )

observation_noise = {
    "screen_image": 0.05 * svi_targets["screen_image"].abs().max(),
    "screen_centroid_x": 5e-5,
    "screen_centroid_y": 5e-5,
}


def svi_optimizer_settings(module_name, parameter_name):
    learning_rate = 5e-5 if ".locs." in parameter_name else 0.03
    return {"lr": learning_rate}


with torch.random.fork_rng(devices=[]):
    torch.manual_seed(0)
    svi_result = infer_model_svi(
        bayesian_model,
        svi_inputs,
        svi_targets,
        observation_noise,
        steps=600,
        num_samples=2000,
        init_scale=2e-4,
        optimizer=pyro.optim.Adam(svi_optimizer_settings),
    )

offset_samples_mm = (
    torch.stack(
        [
            svi_result.posterior_samples[f"model.{latent_name}"].cpu()
            for latent_name in svi_parameter_paths
        ],
        dim=1,
    )
    * 1e3
)
posterior_mean_mm = offset_samples_mm.mean(dim=0)
credible_interval_mm = torch.quantile(
    offset_samples_mm,
    offset_samples_mm.new_tensor([0.025, 0.975]),
    dim=0,
)
for quadrupole_index, quadrupole_name in enumerate(svi_quadrupoles):
    torch.testing.assert_close(
        getattr(reference_model.simulator.segment, quadrupole_name).misalignment,
        true_offsets[quadrupole_index],
    )
    print(
        f"{quadrupole_name} posterior mean [mm]: {posterior_mean_mm[quadrupole_index].tolist()}"
    )
    print(
        f"  95% interval bounds [mm]: {credible_interval_mm[:, quadrupole_index].tolist()}"
    )
    print(f"  Ground truth [mm]: {(true_offsets[quadrupole_index] * 1e3).tolist()}")

In [ ]:
svi_loss_figure, svi_loss_axis = plt.subplots(figsize=(7, 3), layout="constrained")
svi_loss_axis.plot(range(1, len(svi_result.losses) + 1), svi_result.losses)
svi_loss_axis.set(
    xlabel="SVI steps",
    ylabel="Negative ELBO estimate",
    title="Variational fit",
)

svi_figure, svi_axes = plt.subplots(
    len(svi_quadrupoles),
    2,
    figsize=(10, 9),
    layout="constrained",
    squeeze=False,
)
true_offset_mm = true_offsets.detach().cpu() * 1e3
for quadrupole_index, quadrupole_name in enumerate(svi_quadrupoles):
    for coordinate, coordinate_name in enumerate(("Horizontal", "Vertical")):
        axis = svi_axes[quadrupole_index, coordinate]
        samples = offset_samples_mm[:, quadrupole_index, coordinate].numpy()
        lower, upper = credible_interval_mm[:, quadrupole_index, coordinate].tolist()
        axis.hist(samples, bins=40, density=True, alpha=0.7, label="Posterior draws")
        axis.axvspan(lower, upper, color="tab:blue", alpha=0.15, label="95% interval")
        axis.axvline(
            posterior_mean_mm[quadrupole_index, coordinate].item(),
            color="tab:blue",
            label="Posterior mean",
        )
        axis.axvline(
            true_offset_mm[quadrupole_index, coordinate].item(),
            color="tab:red",
            linestyle="--",
            label="Ground truth",
        )
        axis.set(
            xlabel="Offset (mm)",
            ylabel="Density (1/mm)",
            title=f"{quadrupole_name}: {coordinate_name.lower()} misalignment",
        )
        axis.legend(fontsize=8)
svi_figure.suptitle(
    "Approximate posterior conditional on the assumed image and centroid noise"
)
plt.show()

## Predict screen images and centroids from the SVI posterior

Replay the first 128 joint offset draws with `predict_model_svi` at all three quadrupole inputs in `svi_inputs`. Image predictions have shape `(draws, scan_points, height, width)`; each centroid has shape `(draws, scan_points)`. The function also accepts new scan inputs without target outputs.

The top row shows training images and centroids from `svi_targets`. The middle row shows posterior-mean images and centroids, computed by averaging simulator outputs over draws rather than evaluating only the mean offsets. Red crosses mark centroids; the middle row adds marginal one-standard-deviation error bars in x and y. These bars are not joint credible regions. The bottom row shows pixelwise image standard deviation with its own shared color scale.

In [ ]:
from lume_applications.model_calibration.svi import predict_model_svi

svi_prediction_draws = predict_model_svi(
    bayesian_model,
    svi_inputs,
    list(svi_targets),
    svi_result.posterior_samples,
    num_samples=128,
)
svi_prediction_mean = {
    name: draws.mean(dim=0) for name, draws in svi_prediction_draws.items()
}
svi_prediction_std = {
    name: draws.std(dim=0) for name, draws in svi_prediction_draws.items()
}
svi_image_draws = svi_prediction_draws["screen_image"]
svi_image_mean = svi_prediction_mean["screen_image"]
svi_image_std = svi_prediction_std["screen_image"]

svi_x_mm, svi_y_mm = [
    centers.detach().cpu().numpy() * 1e3
    for centers in bayesian_model.model.simulator.segment.SCR1.pixel_bin_centers
]
svi_sample_count = len(svi_inputs["Q1_k"])
svi_shared_max = (
    max(
        float(svi_targets["screen_image"].max()),
        float(svi_image_mean.max()),
    )
    or 1.0
)
uncertainty_max = float(svi_image_std.max()) or 1.0
svi_comparison_figure, svi_comparison_axes = plt.subplots(
    3,
    svi_sample_count,
    figsize=(3 * svi_sample_count, 10),
    layout="constrained",
    squeeze=False,
    sharex=True,
    sharey=True,
)
for sample_index in range(svi_sample_count):
    for row_index, outputs in enumerate((svi_targets, svi_prediction_mean)):
        axis = svi_comparison_axes[row_index, sample_index]
        image = outputs["screen_image"][sample_index].detach().cpu().numpy()
        svi_comparison_artist = axis.pcolormesh(
            svi_x_mm,
            svi_y_mm,
            image,
            shading="nearest",
            cmap="viridis",
            vmin=0,
            vmax=svi_shared_max,
        )
        screen_bounds = svi_comparison_artist.get_datalim(axis.transData)
        axis.set_xlim(screen_bounds.x0, screen_bounds.x1)
        axis.set_ylim(screen_bounds.y0, screen_bounds.y1)
        centroid_x_mm = float(outputs["screen_centroid_x"][sample_index]) * 1e3
        centroid_y_mm = float(outputs["screen_centroid_y"][sample_index]) * 1e3
        centroid_x_std_mm = (
            float(svi_prediction_std["screen_centroid_x"][sample_index]) * 1e3
            if row_index == 1
            else None
        )
        centroid_y_std_mm = (
            float(svi_prediction_std["screen_centroid_y"][sample_index]) * 1e3
            if row_index == 1
            else None
        )
        axis.errorbar(
            centroid_x_mm,
            centroid_y_mm,
            xerr=centroid_x_std_mm,
            yerr=centroid_y_std_mm,
            fmt="+",
            color="tab:red",
            markersize=12,
            markeredgewidth=2,
            capsize=2,
            elinewidth=1,
            label=f"({centroid_x_mm:+.2f}, {centroid_y_mm:+.2f}) mm",
        )
        axis.set_aspect("equal")
        axis.legend(loc="upper right", fontsize=7)
    axis = svi_comparison_axes[2, sample_index]
    uncertainty_artist = axis.pcolormesh(
        svi_x_mm,
        svi_y_mm,
        svi_image_std[sample_index].cpu().numpy(),
        shading="nearest",
        cmap="magma",
        vmin=0,
        vmax=uncertainty_max,
    )
    screen_bounds = uncertainty_artist.get_datalim(axis.transData)
    axis.set_xlim(screen_bounds.x0, screen_bounds.x1)
    axis.set_ylim(screen_bounds.y0, screen_bounds.y1)
    axis.set_aspect("equal")
    svi_comparison_axes[0, sample_index].set_title(
        "\n".join(
            f"{name} = {float(svi_inputs[name][sample_index]):+.2f}"
            for name in ("Q1_k", "Q2_k", "Q3_k")
        )
    )
svi_comparison_axes[0, 0].set_ylabel("Training images")
svi_comparison_axes[1, 0].set_ylabel("SVI posterior mean")
svi_comparison_axes[2, 0].set_ylabel("Posterior std. dev.")
svi_comparison_figure.suptitle(
    r"SVI predictions: centroid crosses and 1-std.-dev. error bars; strengths in $\mathrm{m}^{-2}$"
)
svi_comparison_figure.supxlabel("Horizontal position (mm)")
svi_comparison_figure.supylabel("Vertical position (mm)")
svi_comparison_figure.colorbar(
    svi_comparison_artist,
    ax=list(svi_comparison_axes[:2].flat),
    label="Screen signal",
)
svi_comparison_figure.colorbar(
    uncertainty_artist,
    ax=list(svi_comparison_axes[2]),
    label="Screen signal std. dev.",
)
plt.show()